# Test AgentCore Deployed Runtime

This notebook tests the **deployed** AgentCore Runtime via `invoke-agent-runtime`.

It validates:
1. Text messages through the deployed endpoint
2. Image processing (base64 encoded)
3. Audio transcript processing
4. Video analysis via TwelveLabs Pegasus
5. Memory persistence across invocations (short-term + long-term)

> **Prerequisites**: The `AgentAgentCoreStack` must be deployed successfully.

In [ ]:
%pip install boto3 --quiet

In [ ]:
import os
import botocore.loaders
import botocore.session

# Register the custom 'bedrock-agentcore' service model bundled in the deployment package.
# The system boto3/botocore does not include this service yet.
CUSTOM_DATA_PATH = os.path.abspath(
    os.path.join(
        os.path.dirname("__file__"), "..",
        "00-agent-agentcore", "agent_files", "deployment_package", "botocore", "data",
    )
)

def agentcore_boto3_session(region: str) -> "boto3.Session":
    """Create a boto3 Session that knows about the bedrock-agentcore service."""
    bcore = botocore.session.get_session()
    loader = botocore.loaders.Loader(
        extra_search_paths=[CUSTOM_DATA_PATH],
    )
    bcore.register_component("data_loader", loader)
    return boto3.Session(botocore_session=bcore, region_name=region)

print(f"Custom botocore data path: {CUSTOM_DATA_PATH}")
print("Use agentcore_boto3_session() to create clients for 'bedrock-agentcore'.")

## 1. Setup - Read stack outputs from SSM

In [ ]:
import json
import boto3

REGION = "us-east-1"

ssm = boto3.client("ssm", region_name=REGION)

def get_param(name):
    return ssm.get_parameter(Name=name)["Parameter"]["Value"]

AGENT_RUNTIME_ARN = get_param("/agentcore/agent_runtime_arn")
S3_BUCKET = get_param("/agentcore/s3_bucket_name")
MEMORY_ID = get_param("/agentcore/memory_id")

# Extract runtime ID from ARN (last segment after /runtime/)
AGENT_RUNTIME_ID = AGENT_RUNTIME_ARN.split("/")[-1]

print(f"Runtime ARN:  {AGENT_RUNTIME_ARN}")
print(f"Runtime ID:   {AGENT_RUNTIME_ID}")
print(f"Memory ID:    {MEMORY_ID}")
print(f"S3 Bucket:    {S3_BUCKET}")

## 2. Helper - Invoke Agent Runtime

Wraps the `bedrock-agentcore` data plane `invoke-agent-runtime` API.

- `actor_id` identifies the **user** (long-term memory: facts, preferences)
- `session_id` identifies the **conversation** (short-term memory: turns)

In [ ]:
session = agentcore_boto3_session(REGION)
agentcore = session.client("bedrock-agentcore")

# IDs must be >= 33 chars, padded as the agent expects
TEST_PHONE = "5730012345670000000000"
ACTOR_ID  = f"wa-user-{TEST_PHONE}".ljust(33, "0")   # identifies the USER
SESSION_ID = f"wa-chat-{TEST_PHONE}".ljust(33, "0")   # identifies the CONVERSATION

print(f"actor_id:   '{ACTOR_ID}' ({len(ACTOR_ID)} chars)")
print(f"session_id: '{SESSION_ID}' ({len(SESSION_ID)} chars)")


def invoke_agent(prompt: str, media: dict = None) -> str:
    """Invoke the deployed AgentCore Runtime.
    
    Args:
        prompt: User text message.
        media: Optional media dict (type, format, data, s3_uri).
    
    Returns:
        Agent response text.
    """
    payload = {
        "prompt": prompt,
        "actor_id": ACTOR_ID,
    }
    if media:
        payload["media"] = media

    response = agentcore.invoke_agent_runtime(
        agentRuntimeArn=AGENT_RUNTIME_ARN,
        runtimeSessionId=SESSION_ID,
        runtimeUserId=ACTOR_ID,
        payload=json.dumps(payload).encode("utf-8"),
    )

    content = []
    for chunk in response.get("response", []):
        if isinstance(chunk, bytes):
            content.append(chunk.decode("utf-8"))
        elif isinstance(chunk, dict) and "bytes" in chunk:
            content.append(chunk["bytes"].decode("utf-8"))

    response_text = "".join(content)

    try:
        response_json = json.loads(response_text)
        return response_json.get("result", response_text)
    except json.JSONDecodeError:
        return response_text


print("\nHelper ready.")

## 3. Test - Text Message

In [ ]:
response = invoke_agent("Hola! Mi nombre es Carlos y soy de Colombia. Que puedes hacer por mi?")
print(response)

## 4. Test - Memory Recall (same session)

The agent should remember the name and country from the previous turn.

In [ ]:
response = invoke_agent("Como me llamo y de donde soy?")
print(response)

## 5. Test - Image Processing

Sends a base64-encoded image to the agent via the deployed runtime.

In [ ]:
import base64
from pathlib import Path

IMAGE_PATH = "imagen2.png"

if Path(IMAGE_PATH).exists():
    image_b64 = base64.b64encode(Path(IMAGE_PATH).read_bytes()).decode("utf-8")
    ext = Path(IMAGE_PATH).suffix.lstrip(".")
    fmt = {"jpg": "jpeg", "jpeg": "jpeg", "png": "png", "gif": "gif", "webp": "webp"}.get(ext, "jpeg")

    response = invoke_agent(
        prompt="Describe esta imagen en detalle.",
        media={"type": "image", "format": fmt, "data": image_b64},
    )
    print(response)
else:
    print(f"Image not found: {IMAGE_PATH}")
    print("Place a test image in this directory and update IMAGE_PATH to test.")

## 6. Test - Audio Transcript

In [ ]:
response = invoke_agent(
    prompt="Procesa este audio",
    media={
        "type": "audio_transcript",
        "data": "Necesito agendar una reunion para el martes a las 3 de la tarde con el equipo de marketing para hablar de la estrategia del Q4.",
    },
)
print(response)

## 7. Test - Video Analysis (TwelveLabs Pegasus)

> Upload a test video to S3 first and update `VIDEO_S3_URI`.
> The TwelveLabs Pegasus model (`twelvelabs.pegasus-1-2-v1:0`) must be enabled in Bedrock.

In [ ]:
# Upload the local video to the stack's S3 bucket
s3_client = boto3.client("s3", region_name=REGION)
s3_client.upload_file("Runtime.mp4", S3_BUCKET, "videos/Runtime.mp4")
print(f"Uploaded Runtime.mp4 to s3://{S3_BUCKET}/videos/Runtime.mp4")

VIDEO_S3_URI = f"s3://{S3_BUCKET}/videos/Runtime.mp4"

response = invoke_agent(
    prompt="Analiza este video en detalle.",
    media={"type": "video", "s3_uri": VIDEO_S3_URI},
)
print(response)

## 8. Test - Document Processing

> Place a test PDF in this directory and update `DOCUMENT_PATH`.

In [ ]:
DOCUMENT_PATH = "sample_document.pdf"  # Change this

if Path(DOCUMENT_PATH).exists():
    doc_b64 = base64.b64encode(Path(DOCUMENT_PATH).read_bytes()).decode("utf-8")
    ext = Path(DOCUMENT_PATH).suffix.lstrip(".")
    name = Path(DOCUMENT_PATH).stem

    response = invoke_agent(
        prompt="Resume este documento.",
        media={"type": "document", "format": ext, "data": doc_b64, "name": name},
    )
    print(response)
else:
    print(f"Document not found: {DOCUMENT_PATH}")
    print("Place a test PDF in this directory and update DOCUMENT_PATH to test.")

## 9. Test - Cross-turn Memory

The agent should recall information from all previous interactions in this session.

In [ ]:
# Should recall: name=Carlos, country=Colombia, meeting=Tuesday 3PM marketing Q4
response = invoke_agent("Dame un resumen de todo lo que hemos hablado en esta conversacion.")
print(response)

## 10. Test - Check Memory Records (data plane)

Query the AgentCore Memory directly to inspect what was stored.

In [ ]:
# List memory records for our test actor
# namespace = actor_id prefix used by the agent
try:
    records = agentcore.list_memory_records(
        memoryId=MEMORY_ID,
        namespace=ACTOR_ID,
    )
    memory_records = records.get("memoryRecords", records.get("records", []))
    print(f"Found {len(memory_records)} memory records:\n")
    for record in memory_records:
        print(f"  - Type: {record.get('type', 'N/A')}")
        content = record.get("content", record.get("text", "N/A"))
        print(f"    Content: {str(content)[:200]}")
        print()
except Exception as e:
    print(f"Error querying memory: {e}")